# Transaction Foundation Model on Ray — Part 7: Fine-tune the foundation model

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~10 min (`mini`); ~1–2h at `full`

---

Everything through Part 6 followed NVIDIA's blueprint exactly. This part goes beyond it: NVIDIA never updates the foundation model's weights after pretraining, and we want to know what that leaves on the table.

## Why fine-tune, and what we expect

In Part 6 the foundation model stayed frozen — only XGBoost saw fraud labels. That design is cheap (one pretrained model serves every downstream team) but the model itself never gets to learn what fraud looks like. Fine-tuning changes that: we put a small classification head on the decoder, feed it the labeled training data, and let the labels update the whole network. It's the same move as taking a pretrained LLM and fine-tuning it for a specific task.

We test it two ways, and we'll state our predictions before running anything:

1. **Single transaction.** The head sees exactly what the frozen embedding saw: one tokenized transaction. We expect this to *lose* to Part 6's fusion — the tokenizer is lossy (amounts collapse into 7 buckets, merchants into 2,000 hash buckets), while fusion also gets the exact raw fields.
2. **History window.** The head sees the transaction plus its preceding transactions on the same card. This is the first configuration in the whole series whose inference input includes sequence context — fraud that only looks wrong *in sequence* is invisible to every model so far. This one could beat fusion.

And a third number falls out for free: **fine-tuned + raw** — the fine-tuned model's score added as one extra feature next to the 13 raw fields, using NVIDIA's own fusion recipe. Same logic as Part 6's fusion, stronger ingredient.

Training and evaluation match Part 6 exactly — the same balanced 1M training sample, the same 100K stratified val/test sets, average precision as the score — so every number here lands in the same table as raw, embedding, and fusion.

In [ ]:
import sys, os, json
import logging

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)
ft = cfg["finetune"]
FT_DIR = paths["finetune"]

## Build the labeled token sets

The fine-tune consumes tokens plus labels. For the single-transaction variant this is the same preparation Part 5 ran — the seeded balanced sample and the order-fixing sort on a CPU task, then per-batch tokenization across CPU workers — except the token ids are the product instead of an intermediate on the way to embeddings.

In [ ]:
from src.finetune import build_single_txn_tokens

if not os.path.exists(os.path.join(FT_DIR, "ft_ids_test.npy")):
    stats = build_single_txn_tokens(
        paths["nvsplit"], FT_DIR,
        balanced_train=cfg["embed"]["balanced_train"],   # same sample as Parts 5/6
        max_length=ft["max_length"])
    print(json.dumps(stats, indent=2))
else:
    print("single-transaction tokens present at", FT_DIR)

## Fine-tune with Ray Train

The training function is the same shape as Part 4's: plain PyTorch, wrapped by Ray Train for worker launch, dataset sharding, and DDP. The model is Part 4's pretrained decoder with one new linear layer on top — the last real token's hidden state goes through it to produce a fraud logit. All weights update, not just the head.

Model selection mirrors NVIDIA's downstream recipe: after each epoch we score the val set and keep the weights with the best val average precision, the same role early stopping plays in their XGBoost fit.

In [ ]:
from src.finetune import finetune

sft_dir = os.path.join(FT_DIR, "model_single")
if not os.path.exists(os.path.join(sft_dir, "model.pt")):
    meta = finetune(
        hf_dir=paths["hf"],              # Part 4's pretrained decoder
        tokens_dir=FT_DIR, out_dir=sft_dir, variant="single",
        epochs=ft["epochs"], batch_size=ft["batch_size"], lr=ft["lr"],
        num_workers=ft["num_workers"], use_gpu=ft["use_gpu"],
        storage_base=os.path.join(BASE, "transaction-fm"))
else:
    meta = json.load(open(os.path.join(sft_dir, "meta.json")))
print(f"best epoch {meta['best_epoch']}  val AP {meta['val_ap']:.4f}")

## Score it against Part 6

Same test set, same metric. The table puts the fine-tuned numbers next to what Part 6 measured for raw, embedding, and fusion.

In [ ]:
from src.finetune import score_finetuned

res_single = score_finetuned(paths["hf"], sft_dir, FT_DIR, paths["embeddings"],
                             variant="single", use_gpu=ft["use_gpu"])

ds_metrics = json.load(open(os.path.join(paths["downstream"], "downstream_metrics.json")))
rows = {name: m["pr_auc"] for name, m in ds_metrics["results"].items()}
rows["finetuned (single txn)"] = res_single["finetuned"]["pr_auc"]
rows["finetuned + raw"] = res_single["finetuned_plus_raw"]["pr_auc"]
tbl = pd.Series(rows, name="PR-AUC")
print(tbl.to_string(float_format="%.4f"))

## The history window

Everything so far — the frozen embedding, the single-transaction fine-tune — reads one transaction at a time. But the model *pretrained* on sequences of ~315 transactions, and fraud often only looks wrong in context: the transaction is ordinary, the pattern isn't.

The history variant gives the classifier that context. For each labeled transaction we build a window of it plus up to 18 preceding transactions on the same card, in the exact corpus format the model pretrained on (`<bos> txn <sep> txn ... <eos>`), and pool at the final token — the position that has read the whole window. Building 1.2M windows means re-reading each card's history, so it runs as the same per-card `groupby → map_groups` pipeline as Part 3's corpus build, on CPU workers.

In [ ]:
from src.finetune import build_history_windows

if not os.path.exists(os.path.join(FT_DIR, "ft_hist_ids_test.npy")):
    stats = build_history_windows(
        paths["nvsplit"], paths["source_parquet"], FT_DIR,
        balanced_train=cfg["embed"]["balanced_train"],
        window_txns=ft["window_txns"], window_tokens=ft["window_tokens"])
    print(json.dumps(stats, indent=2))
else:
    print("history windows present at", FT_DIR)

In [ ]:
hist_dir = os.path.join(FT_DIR, "model_history")
if not os.path.exists(os.path.join(hist_dir, "model.pt")):
    meta_h = finetune(
        hf_dir=paths["hf"], tokens_dir=FT_DIR, out_dir=hist_dir, variant="history",
        epochs=ft["epochs"], batch_size=ft["batch_size"], lr=ft["lr"],
        num_workers=ft["num_workers"], use_gpu=ft["use_gpu"],
        storage_base=os.path.join(BASE, "transaction-fm"))
else:
    meta_h = json.load(open(os.path.join(hist_dir, "meta.json")))

res_hist = score_finetuned(paths["hf"], hist_dir, FT_DIR, paths["embeddings"],
                           variant="history", use_gpu=ft["use_gpu"])

rows["finetuned (history)"] = res_hist["finetuned"]["pr_auc"]
rows["finetuned (history) + raw"] = res_hist["finetuned_plus_raw"]["pr_auc"]
tbl = pd.Series(rows, name="PR-AUC")
print(tbl.to_string(float_format="%.4f"))

## Takeaways

- **This part is ours, not NVIDIA's.** The blueprint's comparison (raw / embedding / fusion) is untouched in Part 6; fine-tuning is an extension on top of it, and its numbers are labeled as such.
- **Fine-tuning reuses the whole stack**: Part 4's pretrained weights, Part 5's data preparation, Part 6's evaluation. The new code is a linear head and a training loop — Ray Train runs it with the same `ScalingConfig` knob as pretraining.
- **The history window is the first sequence-aware classifier in the series.** Whether it beats fusion is the interesting result — at `mini` the models are too small to conclude anything; the `full` run's numbers are the answer.
- At `full`, expect the single-transaction fine-tune to trail fusion (the tokenizer sees coarser data than the raw fields) and the history variants to be the ones worth watching.

---

## Next

**Part 8 — Online serving with Ray Serve**: deploy the encoder behind an HTTP endpoint that returns an embedding and a fraud score in one call.